# 🤖 Personal AI Assistant — Colab GPU Server

**Runtime:** GPU → T4 (Runtime › Change runtime type › T4 GPU)

Run every cell from top to bottom. The last cell prints your Cloudflare public URL.

In [ ]:
# ── Cell 1: Mount Google Drive ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── Cell 2: Clone / pull the repo ────────────────────────────────────
import os

REPO_URL = 'https://github.com/Anshulc001/cyber-assistant.git'  # update if repo URL changes
REPO_DIR = '/content/personal-ai-assistant'

if os.path.isdir(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

print('Repo ready at', REPO_DIR)

In [ ]:
# ── Cell 3: Install Python dependencies ──────────────────────────────
!pip install -q -r requirements.txt
print('Dependencies installed.')

In [ ]:
# ── Cell 4: Install cloudflared ───────────────────────────────────────
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
!cloudflared --version

In [ ]:
# ── Cell 5: Set environment variables ────────────────────────────────
import os

os.environ['USE_DRIVE']   = 'true'
os.environ['DEBUG']       = 'false'
os.environ['API_PORT']    = '8000'
os.environ['MODEL_NAME']  = 'Qwythos-9B'   # HuggingFace model ID

print('Environment configured.')

In [ ]:
# ── Cell 6: Pre-load the model ────────────────────────────────────────
# This caches weights in Drive so future sessions are faster.
import sys
sys.path.insert(0, '/content/personal-ai-assistant')

from backend.services.model_service import model_service
model_service.load()
print('Model loaded:', model_service.is_loaded)

In [ ]:
# ── Cell 7: Start FastAPI in background + open Cloudflare tunnel ──────
import subprocess, threading, time, re

# Start uvicorn
server_proc = subprocess.Popen(
    ['uvicorn', 'backend.main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/personal-ai-assistant',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
time.sleep(3)  # let uvicorn bind

# Start cloudflared tunnel
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Parse and print the public URL
public_url = None
deadline = time.time() + 30
while time.time() < deadline:
    line = tunnel_proc.stdout.readline().decode('utf-8', errors='replace')
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print(f'\n✅  Public URL: {public_url}')
    print(f'    Health:     {public_url}/health')
    print(f'    Docs:       {public_url}/docs')
    print('\nPaste this URL into the chat UI and click Connect.')
else:
    print('⚠️  Could not detect tunnel URL. Check tunnel_proc output below:')
    # Print any remaining output for debugging
    for _ in range(10):
        print(tunnel_proc.stdout.readline().decode('utf-8', errors='replace'), end='')

In [ ]:
# ── Cell 8 (optional): Keep-alive loop ────────────────────────────────
# Run this cell to prevent the Colab session from timing out.
# Interrupt the kernel (⏹) to stop it.
import time

print('Keep-alive running. Interrupt to stop.')
while True:
    time.sleep(60)
    print('.', end='', flush=True)